## Applying Split Learning to Chest X-ray Dataset

To adapt the Split Learning framework for a `chest_xray` dataset, we need to make several key modifications, primarily in how data is generated and how the neural network model is structured.

### Data Generation
Instead of generating simple numerical features, the `create_client_dataset` function will now generate synthetic image-like tensors with dimensions appropriate for image processing (e.g., `[BATCH_SIZE, CHANNELS, HEIGHT, WIDTH]`). For demonstration, we will assume grayscale images with `1` channel and a resolution of 64x64 pixels, and `2` output classes (e.g., 'Normal' and 'Pneumonia').

### Model Architecture (Convolutional Neural Network)
The `SplitLearningModel` will be transformed from a purely linear model to a Convolutional Neural Network (CNN). The architecture will consist of five sequential blocks, each comprising:

1.  **Convolutional Layer (`nn.Conv2d`)**: Extracts features from the image.
2.  **ReLU Activation (`nn.ReLU`)**: Introduces non-linearity.
3.  **Max Pooling Layer (`nn.MaxPool2d`)**: Reduces spatial dimensions and provides translation invariance.

After these five convolutional blocks, the output will be flattened and passed to a final linear classification layer (`nn.Linear`) to predict the `NUM_CLASSES`.

**Example Flow (for a 64x64 input image and `HIDDEN_SIZE = 64` initial channels):**

-   **Input**: 1 x 64 x 64 (Channels x Height x Width)
-   **Layer 1**: `Conv2d(1, 64)` -> `MaxPool2d` (Output: 64 x 32 x 32)
-   **Layer 2**: `Conv2d(64, 128)` -> `MaxPool2d` (Output: 128 x 16 x 16)
-   **Layer 3**: `Conv2d(128, 256)` -> `MaxPool2d` (Output: 256 x 8 x 8)
-   **Layer 4**: `Conv2d(256, 512)` -> `MaxPool2d` (Output: 512 x 4 x 4)
-   **Layer 5**: `Conv2d(512, 1024)` -> `MaxPool2d` (Output: 1024 x 2 x 2)
-   **Flatten**: 1024 * 2 * 2 = 4096 features
-   **Classifier**: `Linear(4096, NUM_CLASSES)`

### Split Points
The split points will now occur after each convolutional block. The `client_forward` function will execute a subset of these blocks, and the `server_forward` function will complete the remaining convolutional blocks, flatten the output, and then perform the final classification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
import sympy
import sympy.printing as sympy_printing
import sympy.core as sympy_core_module
import sympy.core.symbol as sympy_symbol_module

# Safely bind imported modules onto sympy namespace attributes
sympy.printing = sympy_printing
sys.modules['sympy.printing'] = sympy_printing
sympy.core = sympy_core_module
sys.modules['sympy.core'] = sympy_core_module
sys.modules['sympy.core.symbol'] = sympy_symbol_module

import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms, datasets

# ============================================================
# GENERAL CONFIGURATION
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Computation Device:", DEVICE)
if DEVICE.type != 'cuda':
    print("\n[WARNING] GPU is not enabled! Please switch your Colab runtime to T4 GPU via Runtime -> Change runtime type.")

CHANNELS = 1
HEIGHT = 64
WIDTH = 64
HIDDEN_SIZE = 64
NUM_CLASSES = 2
NUM_CLIENTS = 5
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
DATA_ROOT_DIR = '/content/drive/MyDrive/Chest_Xray/chest_xray'

# ============================================================
# SPLIT LEARNING ARCHITECTURE
# ============================================================
class SplitLearningModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(CHANNELS, HIDDEN_SIZE, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            ),
            nn.Sequential(
                nn.Conv2d(HIDDEN_SIZE, HIDDEN_SIZE * 2, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            ),
            nn.Sequential(
                nn.Conv2d(HIDDEN_SIZE * 2, HIDDEN_SIZE * 4, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            ),
            nn.Sequential(
                nn.Conv2d(HIDDEN_SIZE * 4, HIDDEN_SIZE * 8, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            ),
            nn.Sequential(
                nn.Conv2d(HIDDEN_SIZE * 8, HIDDEN_SIZE * 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )
        ])

        # Calculate feature spatial map size post 5 pooling operations
        feat_h = HEIGHT // (2 ** len(self.layers))
        feat_w = WIDTH // (2 ** len(self.layers))
        self.flatten_size = (HIDDEN_SIZE * 16) * feat_h * feat_w
        self.classifier = nn.Linear(self.flatten_size, NUM_CLASSES)

    def client_forward(self, x, split_point):
        for i in range(split_point):
            x = self.layers[i](x)
        return x

    def server_forward(self, activation, split_point):
        for i in range(split_point, len(self.layers)):
            activation = self.layers[i](activation)
        activation = activation.view(-1, self.flatten_size)
        return self.classifier(activation)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        x = x.view(-1, self.flatten_size)
        return self.classifier(x)

# ============================================================
# DATASET PREPARATION (80% TRAIN / 20% VALIDATION)
# ============================================================
def get_data_loaders():
    transform = transforms.Compose([
        transforms.Resize((HEIGHT, WIDTH)),
        transforms.Grayscale(num_output_channels=CHANNELS),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    # Load from main train directory
    full_dataset = datasets.ImageFolder(root=f'{DATA_ROOT_DIR}/train', transform=transform)
    total_samples = len(full_dataset)

    # Perform 80% train / 20% validation split
    train_size = int(0.80 * total_samples)
    val_size = total_samples - train_size

    train_dataset, val_dataset = random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    # Distribute 80% train data among simulated clients
    samples_per_client = len(train_dataset) // NUM_CLIENTS
    client_lengths = [samples_per_client] * NUM_CLIENTS
    client_lengths[-1] += len(train_dataset) % NUM_CLIENTS

    client_subsets = random_split(
        train_dataset,
        client_lengths,
        generator=torch.Generator().manual_seed(42)
    )

    client_loaders = [DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True) for ds in client_subsets]
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"Total Dataset Size: {total_samples}")
    print(f"├── 80% Training Split: {train_size} samples across {NUM_CLIENTS} clients (~{samples_per_client} each)")
    print(f"└── 20% Testing Split : {val_size} samples\n")

    return client_loaders, val_loader

# ============================================================
# TRAINING & EVALUATION FUNCTIONS
# ============================================================
def train_model(model, client_loaders, split_points, experiment_name):
    print("=" * 80)
    print(f"TRAINING MODEL: {experiment_name}")
    print(f"Split Configuration across clients: {split_points}")
    print("=" * 80)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    total_communication_bytes = 0
    start_time = time.time()

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for client_id, loader in enumerate(client_loaders):
            split_point = split_points[client_id]

            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()

                # Client phase
                client_activation = model.client_forward(x, split_point)

                # Transmitted tensor setup
                transmitted = client_activation.detach().requires_grad_(True)
                total_communication_bytes += transmitted.numel() * 4  # Float32 = 4 bytes per element

                # Server phase
                output = model.server_forward(transmitted, split_point)
                loss = criterion(output, y)
                loss.backward()

                # Gradient handoff back to client model components
                client_activation.backward(transmitted.grad)
                optimizer.step()

                running_loss += loss.item()

        print(f"Epoch [{epoch + 1}/{EPOCHS}] - Total Split Loss: {running_loss / sum(len(l) for l in client_loaders):.4f}")

    total_time = time.time() - start_time
    total_comm_mb = total_communication_bytes / (1024 * 1024)

    return total_time, total_comm_mb

# Train Baseline with No Split (Entirely processed on one endpoint; 0 communications)
def train_no_split_model(model, client_loaders, experiment_name):
    print("=" * 80)
    print(f"TRAINING MODEL: {experiment_name} (No Split / Centralized)")
    print("=" * 80)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    start_time = time.time()

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for client_id, loader in enumerate(client_loaders):
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()

                # Entire model forward pass on the device
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

        print(f"Epoch [{epoch + 1}/{EPOCHS}] - No Split Loss: {running_loss / sum(len(l) for l in client_loaders):.4f}")

    total_time = time.time() - start_time
    return total_time, 0.0

def evaluate_model(model, val_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    accuracy = (100.0 * correct) / total
    return accuracy

# ============================================================
# MAIN EXECUTION PIPELINE
# ============================================================
if __name__ == "__main__":
    # Prepare data loaders with 80/20 train-val split
    client_loaders, val_loader = get_data_loaders()

    # --------------------------------------------------------
    # MODEL 1: Fixed Split Learning (All clients split at Layer 2)
    # --------------------------------------------------------
    fixed_model = SplitLearningModel().to(DEVICE)
    fixed_splits = [2] * NUM_CLIENTS

    fixed_time, fixed_comm = train_model(
        model=fixed_model,
        client_loaders=client_loaders,
        split_points=fixed_splits,
        experiment_name="FIXED SPLIT LEARNING MODEL"
    )
    fixed_acc = evaluate_model(fixed_model, val_loader)

    print("\n" + "-" * 50)

    # --------------------------------------------------------
    # MODEL 2: Variable Split Learning (Clients split at Layers 1 to 5)
    # --------------------------------------------------------
    variable_model = SplitLearningModel().to(DEVICE)
    variable_splits = [1, 2, 3, 4, 5]

    variable_time, variable_comm = train_model(
        model=variable_model,
        client_loaders=client_loaders,
        split_points=variable_splits,
        experiment_name="VARIABLE SPLIT LEARNING MODEL"
    )
    variable_acc = evaluate_model(variable_model, val_loader)

    print("\n" + "-" * 50)

    # --------------------------------------------------------
    # MODEL 3: No Split / Centralized Baseline Model
    # --------------------------------------------------------
    no_split_model = SplitLearningModel().to(DEVICE)
    no_split_time, no_split_comm = train_no_split_model(
        model=no_split_model,
        client_loaders=client_loaders,
        experiment_name="NO SPLIT BASELINE MODEL"
    )
    no_split_acc = evaluate_model(no_split_model, val_loader)

    # --------------------------------------------------------
    # CONSOLIDATED RESULTS
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL SUMMARY COMPARISON (Tested on 20% Validation Split)")
    print("=" * 80)
    print(f"{'Model Architecture':<30}{'Val Accuracy (%)':<20}{'Training Time (s)':<20}{'Comm Payload (MB)':<20}")
    print("-" * 80)
    print(f"{'Fixed Split Model':<30}{fixed_acc:<20.2f}{fixed_time:<20.2f}{fixed_comm:<20.2f}")
    print(f"{'Variable Split Model':<30}{variable_acc:<20.2f}{variable_time:<20.2f}{variable_comm:<20.2f}")
    print(f"{'No Split Model (Centralized)':<30}{no_split_acc:<20.2f}{no_split_time:<20.2f}{no_split_comm:<20.2f}")
    print("=" * 80)

Computation Device: cuda
Total Dataset Size: 1101
├── 80% Training Split: 880 samples across 5 clients (~176 each)
└── 20% Testing Split : 221 samples

TRAINING MODEL: FIXED SPLIT LEARNING MODEL
Split Configuration across clients: [2, 2, 2, 2, 2]
Epoch [1/5] - Total Split Loss: 0.3003
Epoch [2/5] - Total Split Loss: 0.1387
Epoch [3/5] - Total Split Loss: 0.0954
Epoch [4/5] - Total Split Loss: 0.0554
Epoch [5/5] - Total Split Loss: 0.0553

--------------------------------------------------
TRAINING MODEL: VARIABLE SPLIT LEARNING MODEL
Split Configuration across clients: [1, 2, 3, 4, 5]
Epoch [1/5] - Total Split Loss: 0.6193
Epoch [2/5] - Total Split Loss: 0.2676
Epoch [3/5] - Total Split Loss: 0.2212
Epoch [4/5] - Total Split Loss: 0.1669
Epoch [5/5] - Total Split Loss: 0.1103

--------------------------------------------------
TRAINING MODEL: NO SPLIT BASELINE MODEL (No Split / Centralized)
Epoch [1/5] - No Split Loss: 0.2749
Epoch [2/5] - No Split Loss: 0.1177
Epoch [3/5] - No Split L